In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# VAR Lag Selection
# - input : ./tvpvar_input_scaled.csv
# - output:
#   ./result/lag_selection_table.csv
#   ./result/lag_selection_summary.txt
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_TABLE = RESULT_DIR / "lag_selection_table.csv"
OUT_SUMMARY = RESULT_DIR / "lag_selection_summary.txt"

MAX_LAGS = 10

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

DATE_CANDIDATES = ["Date", "date", "DATE"]


# -----------------------------
# Helpers
# -----------------------------
def find_date_col(df):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None


def mode_smallest(values):
    s = pd.Series(values)
    counts = s.value_counts()
    max_count = counts.max()
    winners = counts[counts == max_count].index.tolist()
    return int(min(winners)), counts.to_dict()


def run_select_order_safely(model, requested_maxlags, min_candidate_lag=1):
    last_error = None

    for maxlags in range(requested_maxlags, min_candidate_lag - 1, -1):
        try:
            sel = model.select_order(maxlags=maxlags)

            aic_list = sel.ics.get("aic", None)
            if aic_list is None:
                continue

            if len(aic_list) >= 2:
                return sel, maxlags

        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(
        f"Unable to run VAR.select_order with lag >= {min_candidate_lag}. "
        f"Last error: {repr(last_error)}"
    )


# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)

date_col = find_date_col(df)
if date_col is not None:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values(date_col).reset_index(drop=True)
    df = df.drop(columns=[date_col])

missing_cols = [c for c in VAR_NAMES if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in input file: {missing_cols}")

df = df[VAR_NAMES].copy()

for c in VAR_NAMES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

before = len(df)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(how="any").reset_index(drop=True)

after = len(df)

if len(df) < 20:
    raise ValueError(f"Too few usable rows after cleaning: {len(df)}")

print("Using columns:", VAR_NAMES)
print("Rows before dropna:", before)
print("Rows after dropna :", after)
print("Dropped rows      :", before - after)
print("Data shape after cleaning:", df.shape)

# -----------------------------
# Lag Selection
# -----------------------------
model = VAR(df)

sel, used_maxlags = run_select_order_safely(
    model,
    requested_maxlags=MAX_LAGS,
    min_candidate_lag=1,
)

ics = sel.ics

available_len = min(
    len(ics["aic"]),
    len(ics["bic"]),
    len(ics["hqic"]),
    len(ics["fpe"]),
)

candidate_lags = list(range(1, available_len))

if len(candidate_lags) == 0:
    raise RuntimeError("No candidate lags >= 1 are available.")

table = pd.DataFrame({
    "lag": candidate_lags,
    "aic": [float(ics["aic"][lag]) for lag in candidate_lags],
    "bic": [float(ics["bic"][lag]) for lag in candidate_lags],
    "hqic": [float(ics["hqic"][lag]) for lag in candidate_lags],
    "fpe": [float(ics["fpe"][lag]) for lag in candidate_lags],
})

best_aic = int(table.loc[table["aic"].idxmin(), "lag"])
best_bic = int(table.loc[table["bic"].idxmin(), "lag"])
best_hqic = int(table.loc[table["hqic"].idxmin(), "lag"])
best_fpe = int(table.loc[table["fpe"].idxmin(), "lag"])

vote_list = [best_aic, best_bic, best_hqic, best_fpe]
recommended_lag, vote_count_dict = mode_smallest(vote_list)

table["best_aic"] = table["lag"] == best_aic
table["best_bic"] = table["lag"] == best_bic
table["best_hqic"] = table["lag"] == best_hqic
table["best_fpe"] = table["lag"] == best_fpe
table["recommended_by_vote"] = table["lag"] == recommended_lag

table.to_csv(OUT_TABLE, index=False, encoding="utf-8-sig")

# -----------------------------
# Summary Text
# -----------------------------
summary_lines = []

summary_lines.append("============================================================")
summary_lines.append("Lag Selection Summary")
summary_lines.append("============================================================")
summary_lines.append("")
summary_lines.append(f"Input file        : {INPUT_FILE}")
summary_lines.append(f"Variables         : {', '.join(VAR_NAMES)}")
summary_lines.append(f"Rows before dropna: {before}")
summary_lines.append(f"Rows after dropna : {after}")
summary_lines.append(f"Dropped rows      : {before - after}")
summary_lines.append(f"Sample size       : {len(df)}")
summary_lines.append(f"Requested maxlags : {MAX_LAGS}")
summary_lines.append(f"Used maxlags      : {used_maxlags}")
summary_lines.append(f"Candidate lags    : 1 ~ {max(candidate_lags)}")
summary_lines.append("")

summary_lines.append("Best lag by criterion")
summary_lines.append("---------------------")
summary_lines.append(f"AIC : {best_aic}")
summary_lines.append(f"BIC : {best_bic}")
summary_lines.append(f"HQIC: {best_hqic}")
summary_lines.append(f"FPE : {best_fpe}")
summary_lines.append("")

summary_lines.append("Vote counts")
summary_lines.append("-----------")
for lag, cnt in sorted(vote_count_dict.items()):
    summary_lines.append(f"lag {lag}: {cnt} vote(s)")
summary_lines.append("")

summary_lines.append(f"Recommended lag (majority vote): {recommended_lag}")
summary_lines.append("")

summary_lines.append("Raw selected_orders from statsmodels")
summary_lines.append("------------------------------------")
for k_name, v in sel.selected_orders.items():
    summary_lines.append(f"{k_name}: {v}")
summary_lines.append("")

summary_lines.append("Decision rule")
summary_lines.append("-------------")
summary_lines.append("- Candidate models were restricted to lag >= 1.")
summary_lines.append("- Final recommendation is based on AIC, BIC, HQIC, and FPE jointly.")
summary_lines.append("- If requested maxlags is infeasible, the code automatically reduces it.")
summary_lines.append("- Compare residual diagnostics for nearby lag candidates if needed.")
summary_lines.append("")

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# -----------------------------
# Print
# -----------------------------
print("\nLag selection table:")
print(table.to_string(index=False))

print("\nBest lag by criterion:")
print(f"AIC : {best_aic}")
print(f"BIC : {best_bic}")
print(f"HQIC: {best_hqic}")
print(f"FPE : {best_fpe}")

print("\nVote counts:")
for lag, cnt in sorted(vote_count_dict.items()):
    print(f"lag {lag}: {cnt} vote(s)")

print(f"\nRecommended lag (majority vote): {recommended_lag}")

print("\nRaw selected_orders:")
for k_name, v in sel.selected_orders.items():
    print(f"{k_name}: {v}")

print("\nSaved:")
print(f" - {OUT_TABLE}")
print(f" - {OUT_SUMMARY}")

Using columns: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']
Rows before dropna: 1032
Rows after dropna : 1032
Dropped rows      : 0
Data shape after cleaning: (1032, 9)

Lag selection table:
 lag       aic       bic      hqic      fpe  best_aic  best_bic  best_hqic  best_fpe  recommended_by_vote
   1 -1.578961 -1.144855 -1.414145 0.206190      True      True       True      True                 True
   2 -1.539836 -0.715034 -1.226685 0.214425     False     False      False     False                False
   3 -1.456318 -0.240821 -0.994833 0.233122     False     False      False     False                False
   4 -1.374334  0.231859 -0.764514 0.253080     False     False      False     False                False
   5 -1.326047  0.670842 -0.567892 0.265670     False     False      False     False                False
   6 -1.245418  1.142166 -0.338928 0.288090     False     False      False     False                Fa